# Corrected CPU CLIP/VLM evaluation

This notebook calls `anomaly-eval run`; it does not maintain a second scoring pipeline.
The shortlist is MobileCLIP2-S2, SigLIP2-B/16-224, SmolVLM-500M and MiniCPM-V 4.0.
VadCLIP is a separately supervised **offline smoking** adaptation, with CLIP zero-shot and linear-probe controls.

Historical 80-image results are development diagnostics. Verified tasks, proxy labels, missing tasks,
frame inputs and chronological past-frame contact sheets stay explicit in the generated report.
No scalar weighted “fitness” combines missing VLM AUC with measured metrics.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import subprocess
import sys

import os
from anomaly_eval.paths import Workspace, configure, resolve_path

# Install this checkout in the notebook kernel with `python -m pip install -e . --no-deps`.
# Server paths come from a config file or CAPSTONE_* environment variables.
WORKSPACE_CONFIG = os.environ.get("CAPSTONE_WORKSPACE_CONFIG")
if WORKSPACE_CONFIG:
    configure(Path(WORKSPACE_CONFIG))
WORKSPACE = Workspace.current()
PREPARED_DIR = None  # optional writable location for a read-only dataset mount
print("Server locations:", WORKSPACE.as_dict())

MODE = "evaluation"  # "quick": small unpaced pilot; "deployment": small paced resource check
LIMIT = None  # override inputs per task/protocol/split; 0 means all
DUTY_CYCLE = 1.0  # below 1 only with MODE="deployment"
MODELS = ["mobileclip2-s2", "siglip2-b16-224", "smolvlm-500m", "minicpm-v4"]
THREADS = 4
MAX_RSS_MB = 24576  # leaves space for the OS on a ~32 GB deployment
CAMERA_FPS = 1.0    # budget only in evaluation/quick; actual pacing in deployment mode
SEED = 42
FETCH_ADDITIONAL = False  # uses public ABODA/CAVIAR; Roboflow needs export credentials/version
FETCH_ORIGINAL = False   # downloads the original shared Drive corpus
RUN_INFERENCE = False    # set True for actual model runs; model weights can be large
RUN_VADCLIP = False      # requires the full original smoking COCO corpus
OFFLINE = False          # True only after weights and dataset are cached
REVIEW_CSV = None        # reviewed new events and matched normals, per docs/evaluation.md
RUN_DIR = WORKSPACE.results_dir / ("notebook_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
print("Results directory:", RUN_DIR)


## Run the shared pipeline

The default prepares and validates data without downloading model weights. Enable `RUN_INFERENCE` to execute comparisons. Every execution uses a new directory.

Evaluation mode runs all prepared inputs without pacing sleeps. Quick mode defaults to 8 inputs per task/split and frame inputs. Deployment mode defaults to 16 inputs per task/split and paces at CAMERA_FPS. LIMIT overrides the sample cap. Smaller cohorts are pilot diagnostics. All five tasks are requested, but only unattended items currently have curated dev/test support; smoking has development data and the other three need reviewed event/normal labels.

In [ ]:
command = [sys.executable, "-m", "anomaly_eval"]
if WORKSPACE_CONFIG:
    command += ["--workspace-config", str(WORKSPACE_CONFIG)]
command += ["run", "--mode", MODE, "--duty-cycle", str(DUTY_CYCLE),
           "--output", str(RUN_DIR), "--models", *MODELS,
           "--threads", str(THREADS), "--max-rss-mb", str(MAX_RSS_MB),
           "--camera-fps", str(CAMERA_FPS), "--seed", str(SEED)]
if LIMIT is not None:
    command += ["--limit", str(LIMIT)]
if FETCH_ADDITIONAL:
    command.append("--fetch")
if FETCH_ORIGINAL:
    command.append("--fetch-drive")
if OFFLINE:
    command.append("--offline")
if REVIEW_CSV:
    command += ["--review-csv", str(REVIEW_CSV)]
if not RUN_INFERENCE:
    command.append("--prepare-only")
if RUN_VADCLIP:
    command.append("--vadclip")
print(" ".join(command))
if PREPARED_DIR:
    command += ["--prepared-dir", str(PREPARED_DIR)]
completed = subprocess.run(command)
if completed.returncode:
    raise RuntimeError(f"Runner exited {completed.returncode}; read {RUN_DIR / 'evaluation.json'} and logs")


In [ ]:
state = json.loads((RUN_DIR / "evaluation.json").read_text())
print("Execution status:", state["status"])
print("All requested tasks have dev/test coverage:", state["full_task_coverage"])
print(json.dumps(state["dataset"]["coverage"], indent=2))
print("Report:", RUN_DIR / "report.md")


In [ ]:
# Each CSV is one matched task / input protocol / split. Do not mix their FPS or pool unsupported tasks.
for record in state["benchmarks"]:
    path = resolve_path(record["output"], RUN_DIR) / "comparison.csv"
    print(record["cohort"], record["split"])
    if not path.is_file():
        print("Incomplete run: inspect logs")
        continue
    with path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            print({key: row.get(key) for key in ("model", "status", "prediction_coverage", "binary_f1",
                                                "roc_auc", "latency_p95_seconds", "peak_rss_mb")})


In [ ]:
for record in state["benchmarks"]:
    directory = resolve_path(record["output"], RUN_DIR)
    for name in ("improvements.json", "calibration.json"):
        path = directory / name
        if path.is_file():
            print(record["cohort"], record["split"], name)
            print(json.dumps(json.loads(path.read_text()), indent=2))


## Reading the results

- Binary recall includes positive abstentions and failures. Coverage and normal unanswered counts remain visible.
- AUPR/AUROC and development-calibrated thresholds require continuous scores; generative hard labels have none.
- Bootstrap samples source recording groups and computes paired differences on exactly shared prediction cohorts.
- CAVIAR intervals are preliminary annotation-based sequence reviews from one staged camera, not deployment-domain evidence.
- Loitering duration and actual rough sleeping need reviewed temporal/event labels. Lying posture or person boxes remain proxies.
- Smoking sessions previously inspected are development data for the zero-shot comparison. VadCLIP has a separately frozen three-way split, but the historically inspected corpus limits external validity.
- p95 and RSS describe this host/configuration. Thread limits cannot emulate the target Ryzen processor. Repeat the same command on the deployment hardware.

See `docs/evaluation.md` for source versions, review schema, rerun commands and remaining coverage gaps.
